### Vorverarbeitung der Daten

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn as skl
import statsmodels.api as sm

df = pd.read_csv("data.csv")
df["date"] = pd.to_datetime(df["DateTime"])

# Tarife werden vorerst zusammengelegt
df = df.groupby("date", as_index=False).agg({
    "KWH": "sum",
    "meters": "sum"
}).reset_index()

# Stromverbrauch eines Verbrauchers jedes Zeitpunktes
df["kwh"] = df["KWH"] / df["meters"]

df = df[["date", "KWH", "kwh", "meters"]]
display(df)

# Erste Visualisierung des Datensatzes als täglicher Durchschnittsverbrauch
series = df.set_index("date")["kwh"].sort_index()
series = series.resample("D").mean()
fig, ax = plt.subplots(figsize=(15, 6))
series.plot(ax=ax, color="#96002A")
ax.set(
    title="Täglicher durchschnittlicher Stromverbrauch",
    xlabel="Datum",
    ylabel="Kilowattstunde",
)
plt.show()

### Datenbereinigung

In [ ]:
display(df.loc[df["KWH"] == 0])
ax = df.loc[df["KWH"] == 0].plot.scatter(x="date", y="KWH")
ax.set_xlim(df["date"].min(), df["date"].max())

prev = df["kwh"].count()
df = df[
    (df["date"].dt.minute % 30 == 0) &
    (df["date"].dt.second == 0)
]
print(prev - df["kwh"].count())

# https://pandas.pydata.org/docs/user_guide/timeseries.html#timeseries-offset-aliases
df = df.set_index("date")[["kwh"]].sort_index().asfreq("30min")
df

# Alternativ
# df = df.dropna()

### Untersuchung der Zeitreihe

In [ ]:
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(15, 6))
ax.margins(x=0)

x = mdates.date2num(df.index)
p = np.poly1d(np.polyfit(x, df["kwh"], 1))
df["kwh_trend"] = p(x)

df["kwh_trendbereinigt"] = df["kwh"] -  df["kwh_trend"]
ax.plot(df.index, df["kwh"], color="#6B001E", alpha=0.7, label="Verbrauch")
ax.plot(df.index, df["kwh_trendbereinigt"], color="#FD8880", alpha=0.7, label="Verbrauch trendbereinigt")
ax.plot(df.index, df["kwh_trend"], color="#000000", alpha=1, label="Trend")
ax.set(
    title="Stromverbrauch eines Verbrauchers",
    xlabel="Datum",
    ylabel="Kilowattstunde",
)
ax.legend()
ax.grid(True, axis="y", alpha=0.5)
plt.show()

In [ ]:
# .hour, .dayofweek, .day
df["focus"] = df.index.hour

seasonal = df.groupby("focus")["kwh_trendbereinigt"].mean()

fig, ax = plt.subplots(figsize=(15, 6))
ax.margins(x=0)

seasonal.plot(ax=ax, color="#9B0000", alpha=0.7, label="Verbrauch")
ax.set(
    title="Saisonalität (Fokus)",
    xlabel="Fokus",
    ylabel="Kilowattstunde",
)
ax.legend()
ax.grid(True, axis="y", alpha=0.5)
plt.show()

df = pd.merge(df, seasonal, on="focus", suffixes=("", "_focus"))
df

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
ax.margins(x=0)

df["kwh_bereinigt"] = df["kwh_trendbereinigt"] - df["kwh_trendbereinigt_focus"]
ax.plot(df.index, df["kwh"], color="#6B001E", alpha=0.5, label="Verbrauch")
ax.plot(df.index, df["kwh_trendbereinigt"], color="#FD8880", alpha=0.5, label="Verbrauch trendbereinigt")
ax.plot(df.index, df["kwh_bereinigt"], color="#FFFC58", alpha=0.7, label="Verbrauch bereinigt")
ax.plot(df.index, df["kwh_trend"], color="#000000FF", alpha=0.7, label="Trend")
ax.plot(df.index, df["kwh_trendbereinigt_focus"], color="#9B0000", alpha=0.7, label="Saisonalität")
ax.set(
    title="Stromverbrauch eines Verbrauchers",
    xlabel="Datum",
    ylabel="Kilowattstunde",
)
ax.set_xlim(pd.Series(df.index).quantile(0.49), pd.Series(df.index).quantile(0.52))
ax.legend()
ax.grid(True, axis="y", alpha=0.5)
plt.show()

In [ ]:
from statsmodels.tsa.seasonal import STL

decomposition = STL(
    df["kwh"],
    period=48  # 48 = Tag, 336 = Woche
)
decomposition = decomposition.fit()
_ = decomposition.plot()

In [ ]:
# from statsmodels.tsa.seasonal import seasonal_decompose

# decomposition = seasonal_decompose(
#     df["kwh"],
#     model="additive",
#     period=48
# )
# _ = decomposition.plot()

In [ ]:
trend = decomposition.trend
seasonal = decomposition.seasonal
resid = decomposition.resid

fig, ax = plt.subplots(4, 1, figsize=(15,6), sharex=True)

ax[0].plot(df["kwh"], color="#6B001E")
ax[0].set_title("Stromverbrauch")

ax[1].plot(trend, color="#FD8880")
ax[1].set_title("Trend")

ax[2].plot(seasonal, color="#00AEC5", alpha=0.7)
ax[2].set_title("Saisonalität")

ax[3].plot(resid, color="#C4C13C")
ax[3].set_title("Residuum")

plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

_ = plot_acf(resid.dropna(), lags=48)
_ = plot_acf(df["kwh_bereinigt"].dropna(), lags=48)
# _ = plot_pacf(resid, lags=48)

### Vorhersage

In [ ]:
from sklearn.linear_model import LinearRegression

# Drop rows with NaN values in the relevant columns for regression
df_x = df.copy()
df_x['U_bereinigt_t-1'] = df_x.kwh_bereinigt.shift(1)
df_x['U_bereinigt_t-2'] = df_x.kwh_bereinigt.shift(2)
df_x['U_bereinigt_t-3'] = df_x.kwh_bereinigt.shift(3)
h_reg = df_x.dropna(subset=['kwh_bereinigt', 'U_bereinigt_t-1', 'U_bereinigt_t-2', 'U_bereinigt_t-3']).copy()

# Define independent variables (X) and dependent variable (y)
X = h_reg[['U_bereinigt_t-1', 'U_bereinigt_t-2', 'U_bereinigt_t-3']]
y = h_reg['kwh_bereinigt']

# Initialize and fit the linear regression model
model = LinearRegression()
model.fit(X, y)

# Predict the values
h_reg['U_bereinigt_pred'] = model.predict(X)

# Merge the predictions back into the original 'h' DataFrame using 'Zeitp' for alignment.
# Note: This will create a new 'U_bereinigt_pred' column in 'h' and will have NaN for rows not in h_reg.
h = pd.merge(df, h_reg['U_bereinigt_pred'], left_index=True, right_index=True)

print("Regression Coefficients:", model.coef_)
print("Regression Intercept:", model.intercept_)
h

In [ ]:
plt.figure(figsize=(15,6))
plt.plot(h.kwh_bereinigt, alpha=0.7, color="red")
plt.plot(h.U_bereinigt_pred, alpha=0.7, color="blue")

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

# TODO
model = ARIMA(df["kwh"], order=(2,1,2)).fit()
forecast = model.forecast(steps=120)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
ax.margins(x=0)

ax.plot(forecast.index, forecast, color="#A5002F75", label="6-Stunden-Vorhersage")
df["kwh"].tail(100).plot(ax=ax, color="#440013", label="Verbrauch")
ax.set(
    title="",
    xlabel="Datum",
    ylabel="Kilowattstunde",
)
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.show()